Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [67]:
NO_RECORDS = 150

In [68]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [69]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [70]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [71]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [72]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234', 'title': 'BREAK2@ESRF', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2348957234</identifier>\n    <datestamp>2026-04-20T06:00:48.078Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2348957234</identifier><creators><creator><creatorName nameType="Personal">JEAN-BAPTISTE JACOB</creatorNa

In [73]:
record_titles

['BREAK2@ESRF',
 'Revealing electrochemically activated polymer infiltration into thick cathodes in solid-state batteries',
 'Characterization of Piezoelectric sensors',
 '3D characterization of deformation mechanisms of the Achilles tendon enthesis using Texture tomography and phase contrast µCT',
 'FInnish National Protein Crystallography Consortium',
 'Multi-scale Multi-techniques investigations of Li-ion batteries: towards a European Battery Hub',
 'Microporosity in mice lacking peripheral circadian clocks in osteocytes',
 'Paramagnons in the extremely overoped cuprate La2-xCaxCuO4',
 'Strain Engineering of Nanoporous Copper for Electrochemical CO2 Reduction',
 'Mechanisms of fast CO2 fixation reaction by enoyl-CoA carboxylases/reductase',
 'Advancing Green Construction through Synchrotron Analysis of Calcined Clay Concrete',
 'PROTEINS RELATED TO PATHOGENESIS OF DISEASES, VIRAL PROTEINS, CELL DIVISION, SIGNALLING AND CHROMATIN PROCESSES',
 'Bluesky and Ibex scanning testing',
 'Co

In [74]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1919247156</identifier>\n    <datestamp>2024-10-10T17:06:57.230Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1919247156</identifier><creators><creator><creatorName nameType="Personal">Jaianth VIJAYAKUMAR</creatorName><givenName>Jaianth</givenName><familyName>Vijayakumar</familyName></creator><creator><creatorName nam

In [75]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-1919247156'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [76]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [77]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['ES-1707', 'XRCT, X-ray Computed Tomography', 'Earth Sciences', 'BM18', 'XRAYS, X-ray Radiation Technique']


In [78]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 121, 122, 123, 124, 125, 126, 127, 128, 129, 131, 132, 133, 134, 135, 136, 137, 138, 140, 141, 142, 144, 145, 146, 147, 148]


In [79]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1584277816', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1919247156', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2008150048', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2232211614', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/514139125', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2231116863', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1432055994', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1830138612', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1901554069', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2228341969', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1059380855', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2170468785', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/404439005', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2328167858', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2297671546', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2012318240', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/231290

In [80]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [81]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Das, Arghadeep', 'name': 'Arghadeep', 'surname': 'Das', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0009-0004-5188-2696'}, 'provenance': None}}, {'fullName': 'Jacob, Jean-Baptiste', 'name': 'Jean-Baptiste', 'surname': 'Jacob', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-2783-4756'}, 'provenance': None}}, {'fullName': 'Miao, Shuting', 'name': 'Shuting', 'surname': 'Miao', 'rank': 3, 'pid': None}, {'fullName': 'Prastyani, Erina', 'name': 'Erina', 'surname': 'Prastyani', 'rank': 4, 'pid': None}, {'fullName': 'Renard, François', 'name': 'François', 'surname': 'Renard', 'rank': 5, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-5125-5930'}, 'provenance': None}}, {'fullName': 'Yang, Hui', 'name': 'Hui', 'surname': 'Yang', 'rank': 6, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-1252-9000'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'langua

In [82]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-2348957234',
  'subjects': ['ES-1707',
   'XRCT, X-ray Computed Tomography',
   'Earth Sciences',
   'BM18',
   'XRAYS, X-ray Radiation Technique']},
 {'doi': '10.15151/esrf-es-2008150048',
  'subjects': ['x-ray probe', 'Life Sciences', 'LS-3372', 'ID19']},
 {'doi': '10.15151/esrf-es-1919247156',
  'subjects': ['x-ray probe', 'BM18', 'IH-MA-552']},
 {'doi': '10.15151/esrf-es-2231116863', 'subjects': ['IH-MD-70', 'ID19']},
 {'doi': '10.15151/esrf-es-2223375524',
  'subjects': ['ID30B',
   'ID23-2',
   'ID30A-3',
   'Macromolecular Crystallography',
   'MX-2735']},
 {'doi': '10.15151/esrf-es-2232211614',
  'subjects': ['ID30B',
   'ID23-2',
   'ID30A-3',
   'Macromolecular Crystallography',
   'MX-2735']},
 {'doi': '10.15151/esrf-es-2223375545',
  'subjects': ['ID30B',
   'ID23-2',
   'ID30A-3',
   'Macromolecular Crystallography',
   'MX-2735']},
 {'doi': '10.15151/esrf-es-1432055994',
  'subjects': ['PaN',
   'HC-5428',
   'Hard Condensed Matter Science',
   

In [83]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234',
  'doi': '10.15151/esrf-es-2348957234'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1584277816',
  'doi': '10.15151/esrf-es-1584277816'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1919247156',
  'doi': '10.15151/esrf-es-1919247156'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2008150048',
  'doi': '10.15151/esrf-es-2008150048'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2232211614',
  'doi': '10.15151/esrf-es-2232211614'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/514139125',
  'doi': '10.15151/esrf-es-514139125'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2231116863',
  'doi': '10.15151/esrf-es-2231116863'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1432055994',
  'doi': '10.15151/esrf-es-1432055994'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1830138612',
  'doi': '10.15151/esrf-es-1830138612'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/190155

In [84]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234',
  'doi': '10.15151/esrf-es-2348957234',
  'enriched_subjects': {'BM18',
   'ES-1707',
   'Earth Sciences',
   'XRAYS, X-ray Radiation Technique',
   'XRCT, X-ray Computed Tomography'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1584277816',
  'doi': '10.15151/esrf-es-1584277816',
  'enriched_subjects': {'Applied Material Science',
   'ID16B',
   'MA-6180',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1919247156',
  'doi': '10.15151/esrf-es-1919247156',
  'enriched_subjects': {'BM18', 'IH-MA-552', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2008150048',
  'doi': '10.15151/esrf-es-2008150048',
  'enriched_subjects': {'ID19', 'LS-3372', 'Life Sciences', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2232211614',
  'doi': '10.15151/esrf-es-2232211614',
  'enriched_subjects': {'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX-273

In [85]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234',
  'doi': '10.15151/esrf-es-2348957234',
  'enriched_subjects': {'BM18',
   'ES-1707',
   'Earth Sciences',
   'XRAYS, X-ray Radiation Technique',
   'XRCT, X-ray Computed Tomography'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1584277816',
  'doi': '10.15151/esrf-es-1584277816',
  'enriched_subjects': {'Applied Material Science',
   'ID16B',
   'MA-6180',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1919247156',
  'doi': '10.15151/esrf-es-1919247156',
  'enriched_subjects': {'BM18', 'IH-MA-552', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2008150048',
  'doi': '10.15151/esrf-es-2008150048',
  'enriched_subjects': {'ID19', 'LS-3372', 'Life Sciences', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2232211614',
  'doi': '10.15151/esrf-es-2232211614',
  'enriched_subjects': {'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX-273

In [86]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234',
  'doi': '10.15151/esrf-es-2348957234',
  'enriched_subjects': {'BM18',
   'ES-1707',
   'Earth Sciences',
   'XRAYS, X-ray Radiation Technique',
   'XRCT, X-ray Computed Tomography'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1584277816',
  'doi': '10.15151/esrf-es-1584277816',
  'enriched_subjects': {'Applied Material Science',
   'ID16B',
   'MA-6180',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1919247156',
  'doi': '10.15151/esrf-es-1919247156',
  'enriched_subjects': {'BM18', 'IH-MA-552', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2008150048',
  'doi': '10.15151/esrf-es-2008150048',
  'enriched_subjects': {'ID19', 'LS-3372', 'Life Sciences', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2232211614',
  'doi': '10.15151/esrf-es-2232211614',
  'enriched_subjects': {'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX-273

In [87]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

138
118


In [88]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [89]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [90]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234',
  'doi': '10.15151/esrf-es-2348957234',
  'enriched_subjects': {'BM18',
   'ES-1707',
   'Earth Sciences',
   'XRAYS, X-ray Radiation Technique',
   'XRCT, X-ray Computed Tomography'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1584277816',
  'doi': '10.15151/esrf-es-1584277816',
  'enriched_subjects': {'Applied Material Science',
   'ID16B',
   'MA-6180',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1919247156',
  'doi': '10.15151/esrf-es-1919247156',
  'enriched_subjects': {'BM18', 'IH-MA-552', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2008150048',
  'doi': '10.15151/esrf-es-2008150048',
  'enriched_subjects': {'ID19', 'LS-3372', 'Life Sciences', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2232211614',
  'doi': '10.15151/esrf-es-2232211614',
  'enriched_subjects': {'ID23-2',
   'ID30A-3',
   'ID30B',
   'MX-273

In [91]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2348957234</identifier>\n    <datestamp>2026-04-20T06:00:48.078Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2348957234</identifier><creators><creator><creatorName nameType="Personal">JEAN-BAPTISTE JACOB</creatorName><givenName>Jean-Baptiste</givenName><familyName>Jacob</familyName><nameIdentifier nameIdentifierSche

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [92]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [93]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [94]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [95]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

22


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234',
  'doi': '10.15151/esrf-es-2348957234',
  'enriched_subjects': {'Beamline'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2328167858',
  'doi': '10.15151/esrf-es-2328167858',
  'enriched_subjects': {'Context (archaeology)', 'Macromolecule'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2297671546',
  'doi': '10.15151/esrf-es-2297671546',
  'enriched_subjects': {'Magnetic nanoparticles', 'Nanoparticle'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2312906786',
  'doi': '10.15151/esrf-es-2312906786',
  'enriched_subjects': {'Strain (injury)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2261325540',
  'doi': '10.15151/esrf-es-2261325540',
  'enriched_subjects': {'Standardization'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313096370',
  'doi': '10.15151/esrf-es-2313096370',
  'enriched_subjects': {'Disinhibition',
   'Hippocampal formation',
   'Inhibitory postsynaptic potential',
   

In [96]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2', 'doi': '10.23662/fors-ds-686-1', 'enriched_subjects': None}]

In [97]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [ ]:
def merge_subjects_from_new_method(base_list, supplement_list):
    """
    Extend enriched_subjects from supplement_list to matching objects in base_list. Matches by id.
    Modifies in-place.
    """
    supplement_index = {item["id"]: item["enriched_subjects"] for item in supplement_list}

    for item in base_list:
        additional_subjects = supplement_index.get(item["id"])

        if additional_subjects:
            # empty set so we can do an union
            current_subjects = item["enriched_subjects"] or set()
            item["enriched_subjects"] = current_subjects | additional_subjects

    return base_list

In [99]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [100]:
# TODO: enriched_subjects should always be a set, empty set if none

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword